# Geovisor - Hospitales y Clinicas ITT

**Proyecto:** Gobierno de Datos 2026

**Modulo:** Espacializacion de Instituciones Prestadoras de Salud

**Repositorio:** [GitHub](https://github.com/j0rg3c45/Hospitales_Cl-nicas_ITT.git)

---

## Objetivo

Generar un visor geografico interactivo (Geovisor) que permita visualizar, filtrar y analizar individualmente cada hospital y clinica del territorio, consumiendo servicios de imagenes satelitales de Google como mapa base.

---
## 1. Instalacion de Dependencias

In [8]:
%pip install folium --quiet

---
## 2. Importacion de Librerias

In [9]:
import json
import folium
from folium import FeatureGroup, LayerControl, GeoJson
from folium.plugins import MiniMap, Fullscreen, LocateControl
from IPython.display import display, HTML

print("Librerias importadas.")
print(f"  Folium: {folium.__version__}")

Librerias importadas.
  Folium: 0.20.0


---
## 3. Datos de Hospitales y Clinicas

In [10]:
# Datos de puntos (marcadores)
features = [
    {
        "nombre": "Hospital Terron Colorado - ESE Ladera",
        "direccion": "Av. 4a Oe.",
        "comuna": "COMUNA 1",
        "municipio": "Cali",
        "departamento": "Valle del Cauca",
        "tipo_proyecto": "Infraestructura y dotacion hospitalaria",
        "usuarios": 91545,
        "servicios_anuales": 304563,
        "valor_total": "$29,199,204,978",
        "inversion_obra": "$25,445 M",
        "inversion_interventoria": "$1,781 M",
        "inversion_dotacion": "$1,972 M",
        "capacidad_actual": 27,
        "capacidad_proyectada": 42,
        "lat": 3.4524408768046295,
        "lon": -76.55915347465202
    },
    {
        "nombre": "Centro De Salud La Rivera",
        "direccion": "Cra. 1g #65-35",
        "comuna": "Comuna 5",
        "municipio": "Cali",
        "departamento": "Valle del Cauca",
        "lat": 3.474079476955456,
        "lon": -76.49050412813828
    },
    {
        "nombre": "IPS Antonio Narino",
        "direccion": "Cra. 40 #38-99",
        "comuna": "Comuna 16",
        "municipio": "Cali",
        "departamento": "Valle del Cauca",
        "tipo_proyecto": "Infraestructura y servicios de salud integral",
        "consulta_externa": "Enfermeria (4), Medicina General (4), Medicina Interna (1), Odontologia (2), Nutricion (1), Pediatria (1), Psicologia (1), Vacunacion (1)",
        "apoyo_diagnostico": "Lab. Clinico (2 cubiculos), Terapia Ocupacional, Terapia Respiratoria, Fisioterapia (4 camillas), Fonoaudiologia, Citologias",
        "terapias": "Comportamentales ABA, Rehabilitacion Fisica, Conductuales, Cognitivas, Estimulacion Neuropsicologica, Neurodesarrollo",
        "cobertura": "Educacion, Cultura, Orientaciones, Deporte, Alimentacion, Transporte, Talleres prevocacionales",
        "lat": 3.4162184243856415,
        "lon": -76.5083103585888
    }
]

# Poligonos de terrenos (convertidos a EPSG:4326)
poligonos = [
    {
        "nombre": "Terreno La Rivera - Comuna 05",
        "coords": [
            [3.4740086, -76.4902269], [3.4738546, -76.4907211], [3.4751942, -76.4911306],
            [3.4754007, -76.4902942], [3.4754079, -76.4902657], [3.4754107, -76.4902446],
            [3.4754059, -76.4902093], [3.4753945, -76.4901825], [3.4753789, -76.4901621],
            [3.4753648, -76.4901466], [3.4753463, -76.490128], [3.4753252, -76.4901113],
            [3.4752917, -76.4900927], [3.4752423, -76.4900744], [3.4751737, -76.4900498],
            [3.4750354, -76.4900015], [3.4746139, -76.4898702], [3.4741893, -76.489738],
            [3.474179, -76.4897356], [3.4741701, -76.4897351], [3.4741618, -76.489738],
            [3.4741568, -76.4897475], [3.474153, -76.4897619], [3.4741441, -76.489792],
            [3.4740086, -76.4902269]
        ]
    },
    {
        "nombre": "Terreno Antonio Narino - Comuna 16",
        "coords": [
            [3.4165601, -76.5092721], [3.4171388, -76.5086692], [3.4158096, -76.5074186],
            [3.415249, -76.5080124], [3.4165601, -76.5092721]
        ]
    }
]

print(f"Datos cargados: {len(features)} instituciones, {len(poligonos)} terrenos.")
for i, f in enumerate(features):
    print(f"  {i+1}. {f['nombre']} ({f['lat']:.4f}, {f['lon']:.4f})")
for i, p in enumerate(poligonos):
    print(f"  Terreno {i+1}: {p['nombre']}")

Datos cargados: 3 instituciones, 2 terrenos.
  1. Hospital Terron Colorado - ESE Ladera (3.4524, -76.5592)
  2. Centro De Salud La Rivera (3.4741, -76.4905)
  3. IPS Antonio Narino (3.4162, -76.5083)
  Terreno 1: Terreno La Rivera - Comuna 05
  Terreno 2: Terreno Antonio Narino - Comuna 16


---
## 4. Espacializacion / Geovisor

Se construye el mapa interactivo con:

- Mapa base: Google Satellite, Google Hybrid, Google Maps, OpenStreetMap.
- Capas individuales: cada hospital/clinica como capa independiente.
- Poligonos de terrenos: contorno rojo delgado, sin relleno.
- Layer Control: permite encender/apagar cada institucion y terreno.
- Popups informativos con datos de cada IPS.

In [11]:
# Tiles de Google
GOOGLE_SATELLITE = "https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}"
GOOGLE_MAPS = "https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}"
GOOGLE_HYBRID = "https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}"

# Centro del mapa
center_lat = sum(f['lat'] for f in features) / len(features)
center_lon = sum(f['lon'] for f in features) / len(features)

# Crear mapa
mapa = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=13,
    tiles=None,
    control_scale=True
)

# Basemaps
folium.TileLayer(tiles=GOOGLE_SATELLITE, attr="Google", name="Google Satellite", overlay=False).add_to(mapa)
folium.TileLayer(tiles=GOOGLE_HYBRID, attr="Google", name="Google Hybrid", overlay=False).add_to(mapa)
folium.TileLayer(tiles=GOOGLE_MAPS, attr="Google", name="Google Maps", overlay=False).add_to(mapa)
folium.TileLayer(tiles="OpenStreetMap", name="OpenStreetMap", overlay=False).add_to(mapa)

# --- Capas de marcadores (puntos) ---
colores = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'darkblue', 'darkgreen']

for idx, f in enumerate(features):
    color = colores[idx % len(colores)]
    fg = FeatureGroup(name=f['nombre'], show=True)

    popup_html = (
        f"<div style='font-family:Arial;font-size:11px;min-width:250px;max-height:400px;overflow-y:auto;'>"
        f"<h4 style='margin-bottom:5px;color:#333;'>{f['nombre']}</h4>"
        f"<hr style='margin:3px 0;'>"
        f"<b>Direccion:</b> {f['direccion']}<br>"
        f"<b>Comuna:</b> {f['comuna']}<br>"
        f"<b>Municipio:</b> {f['municipio']}<br>"
        f"<b>Departamento:</b> {f['departamento']}<br>"
    )

    if 'valor_total' in f:
        popup_html += (
            f"<hr style='margin:3px 0;'>"
            f"<b>Tipo:</b> {f['tipo_proyecto']}<br>"
            f"<b>Usuarios:</b> {f['usuarios']:,}<br>"
            f"<b>Servicios/anio:</b> {f['servicios_anuales']:,}<br>"
            f"<b>Valor total:</b> {f['valor_total']}<br>"
            f"<b>Obra:</b> {f['inversion_obra']}<br>"
            f"<b>Interventoria:</b> {f['inversion_interventoria']}<br>"
            f"<b>Dotacion:</b> {f['inversion_dotacion']}<br>"
            f"<b>Capacidad actual:</b> {f['capacidad_actual']} ambientes<br>"
            f"<b>Capacidad proyectada:</b> {f['capacidad_proyectada']} ambientes<br>"
        )

    if 'consulta_externa' in f:
        popup_html += (
            f"<hr style='margin:3px 0;'>"
            f"<b>Tipo:</b> {f['tipo_proyecto']}<br>"
            f"<b>Consulta Externa:</b> {f['consulta_externa']}<br>"
            f"<b>Apoyo Diagnostico:</b> {f['apoyo_diagnostico']}<br>"
            f"<b>Terapias:</b> {f['terapias']}<br>"
            f"<b>Cobertura:</b> {f['cobertura']}<br>"
        )

    popup_html += (
        f"<hr style='margin:3px 0;'>"
        f"<b>Lat:</b> {f['lat']:.6f}<br>"
        f"<b>Lon:</b> {f['lon']:.6f}<br>"
        f"</div>"
    )

    folium.Marker(
        location=[f['lat'], f['lon']],
        popup=folium.Popup(popup_html, max_width=350),
        tooltip=f['nombre'],
        icon=folium.Icon(color=color, icon='plus-sign', prefix='glyphicon')
    ).add_to(fg)

    fg.add_to(mapa)

# --- Capas de poligonos (terrenos) ---
# Contorno rojo delgado, sin relleno
for poly in poligonos:
    fg_poly = FeatureGroup(name=poly['nombre'], show=True)

    # Convertir a formato GeoJSON [lon, lat]
    geojson_coords = [[coord[1], coord[0]] for coord in poly['coords']]

    geojson_feature = {
        "type": "Feature",
        "properties": {"nombre": poly['nombre']},
        "geometry": {
            "type": "Polygon",
            "coordinates": [geojson_coords]
        }
    }

    GeoJson(
        geojson_feature,
        style_function=lambda x: {
            'color': 'red',
            'weight': 2,
            'fillOpacity': 0,
            'fillColor': 'transparent'
        },
        tooltip=poly['nombre']
    ).add_to(fg_poly)

    fg_poly.add_to(mapa)

# --- Controles ---
LayerControl(collapsed=False, position='topright').add_to(mapa)
MiniMap(toggle_display=True, position='bottomleft').add_to(mapa)
Fullscreen(position='topleft', title='Pantalla completa', title_cancel='Salir').add_to(mapa)
LocateControl(strings={'title': 'Mi ubicacion'}).add_to(mapa)

print(f"Geovisor construido: {len(features)} marcadores + {len(poligonos)} terrenos.")
print(f"Centro: [{center_lat:.4f}, {center_lon:.4f}]")

Geovisor construido: 3 marcadores + 2 terrenos.
Centro: [3.4476, -76.5193]


---
## 5. Visualizacion

In [12]:
display(mapa)

---
## 6. Exportacion

In [13]:
OUTPUT_FILE = "geovisor_hospitales_clinicas_ITT.html"
mapa.save(OUTPUT_FILE)
print(f"Mapa exportado: {OUTPUT_FILE}")

try:
    from google.colab import files
    files.download(OUTPUT_FILE)
except:
    print("Abra el archivo desde el explorador de archivos.")

Mapa exportado: geovisor_hospitales_clinicas_ITT.html


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>